# FEATURE ENGINEERING & SELECTION

In [1]:

import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif


### 1. Load dataset

In [2]:
df = pd.read_csv("seattle-weather.csv")

print("Original shape:", df.shape)
print("\nOriginal columns:")
print(df.columns.tolist())


Original shape: (1461, 6)

Original columns:
['date', 'precipitation', 'temp_max', 'temp_min', 'wind', 'weather']



### 2. Convert date safely

In [3]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

### 3. Feature Engineering

In [4]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek

In [5]:
# Temperature-based features

df["temp_avg"] = (df["temp_max"] + df["temp_min"]) / 2
df["temp_range"] = df["temp_max"] - df["temp_min"]

### 4. Remove rows with missing target/features needed for selection

In [6]:
df = df.dropna(subset=["weather"])

### 5. Select numeric features only

In [7]:
feature_columns = [
    "precipitation",
    "temp_max",
    "temp_min",
    "wind",
    "year",
    "month",
    "day",
    "day_of_week",
    "temp_avg",
    "temp_range"
]

In [8]:
feature_columns = [c for c in feature_columns if c in df.columns]

X = df[feature_columns].apply(pd.to_numeric, errors="coerce")
y = df["weather"]

# Fill any numeric missing values
X = X.fillna(X.median())

print("\nFeatures before selection:")
print(X.columns.tolist())


Features before selection:
['precipitation', 'temp_max', 'temp_min', 'wind', 'year', 'month', 'day', 'day_of_week', 'temp_avg', 'temp_range']


### 6. Feature Selection using SelectKBest

In [9]:
k = min(7, X.shape[1])

selector = SelectKBest(score_func=f_classif, k=k)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("\nSelected Features:")
print(list(selected_features))

# Feature scores
scores = pd.DataFrame({
    "Feature": X.columns,
    "Score": selector.scores_
}).sort_values("Score", ascending=False)

print("\nFeature Scores:")
print(scores.to_string(index=False))

# Final selected dataset
X_selected_df = pd.DataFrame(X_selected, columns=selected_features, index=X.index)

print("\nFinal selected feature shape:", X_selected_df.shape)
print("\nFinal selected data:")
print(X_selected_df.head())

# Save for Model Selection notebook
final_model_data = X_selected_df.copy()
final_model_data["weather"] = y

final_model_data.to_csv("weather_features_selected.csv", index=False)

print("\nSaved successfully as: weather_features_selected.csv")



Selected Features:
['precipitation', 'temp_max', 'temp_min', 'wind', 'year', 'temp_avg', 'temp_range']

Feature Scores:
      Feature      Score
   temp_range 185.231665
precipitation 119.086023
     temp_max  94.838655
     temp_avg  63.183454
         wind  40.779274
         year  32.612027
     temp_min  30.281944
        month   3.284632
          day   1.080409
  day_of_week   0.241695

Final selected feature shape: (1461, 7)

Final selected data:
   precipitation  temp_max  temp_min  wind    year  temp_avg  temp_range
0            0.0      12.8       5.0   4.7  2012.0      8.90         7.8
1           10.9      10.6       2.8   4.5  2012.0      6.70         7.8
2            0.8      11.7       7.2   2.3  2012.0      9.45         4.5
3           20.3      12.2       5.6   4.7  2012.0      8.90         6.6
4            1.3       8.9       2.8   6.1  2012.0      5.85         6.1

Saved successfully as: weather_features_selected.csv
